# 통계 검정

## 학습 목표

1. 이전 차시에서 한 **가설검정**의 기본 개념과 p-value를 실제로 활용해보기 위한 단계
2. **t-검정, ANOVA**로 집단 간 평균을 비교함
3. **카이제곱 검정**으로 범주형 변수의 관계를 분석함
4. **비모수 검정**의 활용 상황을 판단함


## 실습 데이터셋

| 데이터셋 | 출처 | 용도 |
|----------|------|------|
| **Iris** | sklearn | 품종별 꽃잎 길이 비교 |
| **Titanic** | seaborn | 성별-생존 관계 분석 |
| **Wine Quality** | UCI | 품질 등급별 알코올 함량 비교 |

## 환경 설정

실습에 필요한 라이브러리를 import함. scipy.stats 모듈에서 다양한 통계 검정 함수를 제공함.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (ttest_ind, ttest_rel, f_oneway,
                         chi2_contingency, mannwhitneyu,
                         wilcoxon, kruskal, shapiro)
from sklearn.datasets import load_iris

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 경고 메시지 숨기기
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 로드 완료!")

라이브러리 로드 완료!


## 데이터 로드

In [2]:
# 1) Iris 데이터셋 (붓꽃 데이터)
iris_data = load_iris()
iris = pd.DataFrame(iris_data.data, columns=['sepal_length', 'sepal_width',
                                              'petal_length', 'petal_width'])
iris['species'] = pd.Categorical.from_codes(iris_data.target,
                                             ['setosa', 'versicolor', 'virginica'])

print(f"[1] Iris 데이터셋 로드 완료")
print(f"    - 샘플 수: {len(iris)}")
print(f"    - 품종: {iris['species'].unique().tolist()}")

# 2) Titanic 데이터셋 (타이타닉 생존 데이터)
titanic = sns.load_dataset('titanic')
print(f"\n[2] Titanic 데이터셋 로드 완료")
print(f"    - 샘플 수: {len(titanic)}")
print(f"    - 주요 변수: sex, survived, pclass, age, fare")

# 3) Wine Quality 데이터셋 (와인 품질 데이터)
wine_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
wine = pd.read_csv(wine_url, sep=';')
print(f"\n[3] Wine Quality 데이터셋 로드 완료")
print(f"    - 샘플 수: {len(wine)}")
print(f"    - 품질 등급: {sorted(wine['quality'].unique())}")

[1] Iris 데이터셋 로드 완료
    - 샘플 수: 150
    - 품종: ['setosa', 'versicolor', 'virginica']

[2] Titanic 데이터셋 로드 완료
    - 샘플 수: 891
    - 주요 변수: sex, survived, pclass, age, fare

[3] Wine Quality 데이터셋 로드 완료
    - 샘플 수: 1599
    - 품질 등급: [np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


## 가설검정이란?

데이터를 바탕으로 주장을 검증하는 절차임.

```mermaid
flowchart TB
    subgraph 가설설정["가설 설정"]
        H0[귀무가설 H0<br/>차이가 없다<br/>효과가 없다]
        H1[대립가설 H1<br/>차이가 있다<br/>효과가 있다]
    end

    subgraph 예시["제조업 예시"]
        EX1[H0: A라인과 B라인<br/>불량률은 같다]
        EX2[H1: A라인과 B라인<br/>불량률은 다르다]
    end

    H0 --> |"기본 가정"| CHECK[데이터 수집<br/>및 분석]
    H1 --> |"입증 목표"| CHECK

    CHECK --> RESULT{p-value<br/>< 0.05?}

    RESULT --> |"Yes"| REJECT[H0 기각<br/>H1 채택<br/>통계적 유의]
    RESULT --> |"No"| ACCEPT[H0 유지<br/>유의하지 않음]
```

## p-value란?

귀무가설이 참일 때, 관측 결과가 나올 확률임.

```mermaid
flowchart TB
    PVALUE[p-value란?<br/>귀무가설이 참일 때<br/>관측 결과가 나올 확률]

    PVALUE --> RANGE[p-value 범위별 해석]

    RANGE --> R1["p < 0.001<br/>매우 강력한 증거"]
    RANGE --> R2["p < 0.01<br/>강력한 증거"]
    RANGE --> R3["p < 0.05<br/>유의한 증거"]
    RANGE --> R4["p >= 0.05<br/>유의하지 않음"]

    subgraph 올바른해석["올바른 해석"]
        C1[H0가 참일 때<br/>이 결과가 나올 확률]
        C2[증거의 강도 측정]
    end

    subgraph 잘못된해석["잘못된 해석"]
        W1[H0가 참일 확률 X]
        W2[H1이 참일 확률 X]
        W3[효과의 크기 X]
    end
```

### p-value 해석 주의사항

| 잘못된 해석 | 올바른 해석 |
|------------|------------|
| "H0가 참일 확률" | H0가 참일 때 이 결과가 나올 확률 |
| "H1이 참일 확률" | 결과가 H0와 얼마나 상반되는지 척도 |
| "효과의 크기" | **통계적 유의성만 측정, 효과 크기 아님** |

In [4]:
# Iris에서 setosa와 versicolor의 꽃잎 길이(petal_length) 비교
setosa_petal = iris[iris['species'] == 'setosa']['petal_length']
versicolor_petal = iris[iris['species'] == 'versicolor']['petal_length']

print(f"Setosa 꽃잎 길이:")
print(f"  - 평균: {setosa_petal.mean():.2f} cm")
print(f"  - 표준편차: {setosa_petal.std():.2f} cm")
print(f"  - 샘플 수: {len(setosa_petal)}")

print(f"\nVersicolor 꽃잎 길이:")
print(f"  - 평균: {versicolor_petal.mean():.2f} cm")
print(f"  - 표준편차: {versicolor_petal.std():.2f} cm")
print(f"  - 샘플 수: {len(versicolor_petal)}")

# 평균 차이
mean_diff = versicolor_petal.mean() - setosa_petal.mean()
print(f"\n평균 차이: {mean_diff:.2f} cm")

Setosa 꽃잎 길이:
  - 평균: 1.46 cm
  - 표준편차: 0.17 cm
  - 샘플 수: 50

Versicolor 꽃잎 길이:
  - 평균: 4.26 cm
  - 표준편차: 0.47 cm
  - 샘플 수: 50

평균 차이: 2.80 cm


In [5]:
# t-검정 수행하는 부분!
t_stat, p_value = stats.ttest_ind(setosa_petal, versicolor_petal) 

print(f"\nt-검정 결과:")
print(f"  - t 통계량: {t_stat:.4f}")
print(f"  - p-value: {p_value:.2e}")

# 해석
alpha = 0.05
if p_value < alpha:
    print(f"\n[결론] p-value({p_value:.2e}) < alpha({alpha})")
    print("       귀무가설 기각: 두 품종의 꽃잎 길이에 통계적으로 유의한 차이가 있음")
else:
    print(f"\n[결론] p-value({p_value:.4f}) >= alpha({alpha})")
    print("       귀무가설 채택: 두 품종의 꽃잎 길이에 유의한 차이 없음")


t-검정 결과:
  - t 통계량: -39.4927
  - p-value: 5.40e-62

[결론] p-value(5.40e-62) < alpha(0.05)
       귀무가설 기각: 두 품종의 꽃잎 길이에 통계적으로 유의한 차이가 있음


## 효과 크기 (Effect Size)

### p-value의 한계

p-value는 **차이의 유무**만 알려주고, **차이의 크기**는 알려주지 않음.

표본 크기가 크면 작은 차이도 통계적으로 유의미하게 나타날 수 있음. "통계적으로 유의미"가 "실질적으로 의미있음"과 동일하지 않음.

### Cohen's d

**Cohen's d**는 두 그룹 간 평균 차이를 표준편차 단위로 표현한 효과 크기임.

$$d = \frac{\bar{X}_1 - \bar{X}_2}{s_p}$$

여기서 합동표준편차:

$$s_p = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1 + n_2 - 2}}$$

### Cohen's d 해석 기준

| 절대값 d | 해석 |
|---------|------|
| < 0.2 | 작은 효과 (Small) |
| 0.2 ~ 0.5 | 작은~중간 효과 |
| 0.5 ~ 0.8 | 중간 효과 (Medium) |
| > 0.8 | 큰 효과 (Large) |

In [33]:
def cohens_d(group1, group2):
    """
    Cohen's d 효과 크기 계산
    
    Parameters
    ----------
    group1, group2 : array-like
        비교할 두 그룹의 데이터
        
    Returns
    -------
    d : float
        Cohen's d 효과 크기
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    
    # 합동 표준편차
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1 + n2 - 2))
    
    # Cohen's d
    d = (np.mean(group1) - np.mean(group2)) / pooled_std
    
    return d


# 효과 크기 계산
d = cohens_d(setosa_petal, versicolor_petal)

print("효과 크기 분석")
print("=" * 50)
print(f"Cohen's d = {d:.4f}")

# 해석
if abs(d) < 0.2:
    interpretation = "작은 효과 (Small)"
elif abs(d) < 0.5:
    interpretation = "작은~중간 효과"
elif abs(d) < 0.8:
    interpretation = "중간 효과 (Medium)"
else:
    interpretation = "큰 효과 (Large)"

print(f"해석: {interpretation}")
print(f"\n의미: setosa종과 versicolor의 꽃잎 크기의 차이는versicolor")
print(f"      합동표준편차의 약 {abs(d):.2f}배에 해당함")

효과 크기 분석
Cohen's d = -7.8985
해석: 큰 효과 (Large)

의미: setosa종과 versicolor의 꽃잎 크기의 차이는
      합동표준편차의 약 7.90배에 해당함


**단, 실제로는 T-TEST등 각종 TEST들을 수행하기 위해서는 원본 데이터의 분포, 성질이 몇가지 조건을 만족해야 함**

## 가설검정 5단계 절차 요약(핵심)

```mermaid
flowchart TB
    S1[1단계: 가설 설정<br/>H0: 두 그룹 평균 동일<br/>H1: 두 그룹 평균 다름]

    S1 --> S2[2단계: 유의수준 결정<br/>alpha = 0.05<br/>통상적 기준]

    S2 --> S3[3단계: 검정 통계량 계산<br/>t-value, F-value<br/>chi-square 등]

    S3 --> S4[4단계: p-value 계산<br/>scipy.stats 활용]

    S4 --> S5{5단계: 결론 도출}

    S5 --> |"p < alpha"| REJECT[H0 기각<br/>통계적으로<br/>유의한 차이]
    S5 --> |"p >= alpha"| ACCEPT[H0 유지<br/>유의한 차이 없음]
```

# 검정 선택 가이드

## 상황별 적절한 검정 방법

```mermaid
flowchart TB
    START[데이터 유형<br/>확인]

    START --> TYPE{변수 유형?}

    TYPE --> |"연속형"| CONT[연속형 변수<br/>평균 비교]
    TYPE --> |"범주형"| CAT[범주형 변수<br/>빈도 비교]

    CONT --> GROUPS{비교 그룹 수?}

    GROUPS --> |"2개"| TWO[2개 그룹]
    GROUPS --> |"3개 이상"| THREE[3개 이상 그룹]

    TWO --> PAIR{같은 대상?}
    PAIR --> |"Yes"| PAIRED_T[대응표본<br/>ttest_rel]
    PAIR --> |"No"| IND_T[독립표본<br/>ttest_ind]

    THREE --> NORM1{정규분포?}
    NORM1 --> |"Yes"| ANOVA[ANOVA<br/>f_oneway]
    NORM1 --> |"No"| KW[Kruskal-Wallis<br/>kruskal]

    IND_T --> NORM2{정규분포?}
    NORM2 --> |"Yes"| KEEP_T[t-검정 유지]
    NORM2 --> |"No"| MW[Mann-Whitney U<br/>mannwhitneyu]

    PAIRED_T --> NORM3{정규분포?}
    NORM3 --> |"Yes"| KEEP_PT[대응 t 유지]
    NORM3 --> |"No"| WILC[Wilcoxon<br/>wilcoxon]

    CAT --> CHI[카이제곱 검정<br/>chi2_contingency]
```

# 모수 검정

## 2.1 모수 검정이란?

### 개념 설명

정규분포를 가정하는 검정 방법임.

**전제 조건:**
- 데이터가 **정규분포**를 따름
- 모집단의 **모수**(평균, 분산)를 추정

**대표적 검정 방법:**

| 검정 | 목적 | 비교 그룹 수 |
|------|------|-------------|
| **독립표본 t-검정** | 두 그룹 평균 비교 | 2개 (독립) |
| **대응표본 t-검정** | 전후 평균 비교 | 2개 (쌍) |
| **일원분산분석 (ANOVA)** | 여러 그룹 평균 비교 | 3개 이상 |

## t-검정 (t-test)

### t-검정의 종류

| 종류 | 상황 | 예시 |
|------|------|------|
| **단일표본 t-검정** | 표본평균과 알려진 모평균 비교 | 생산량이 목표(1200개)와 다른가? |
| **독립표본 t-검정** | 두 독립 그룹의 평균 비교 | 라인 A vs 라인 B |
| **대응표본 t-검정** | 동일 대상의 전후 비교 | 개선 전 vs 개선 후 |

### t-통계량

#### 독립표본 t-검정 (등분산 가정)

$$t = \frac{\bar{X}_1 - \bar{X}_2}{s_p \sqrt{\frac{1}{n_1} + \frac{1}{n_2}}}$$

여기서 합동표준편차(pooled standard deviation):

$$s_p = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1 + n_2 - 2}}$$

자유도(degrees of freedom):

$$df = n_1 + n_2 - 2$$

#### Welch's t-검정 (등분산 가정 불필요)

$$t = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}$$

자유도 (Welch-Satterthwaite 근사):

$$df = \frac{\left(\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}\right)^2}{\frac{(s_1^2/n_1)^2}{n_1-1} + \frac{(s_2^2/n_2)^2}{n_2-1}}$$

### t-검정의 가정

| 가정 | 내용 | 확인 방법 |
|------|------|----------|
| **독립성** | 두 그룹이 서로 독립 | 실험 설계 확인 |
| **정규성** | 각 그룹이 정규분포 | Shapiro-Wilk, Q-Q plot |
| **등분산성** | 두 그룹의 분산이 동일 | Levene's test |

**실무 팁**:
- 표본 크기가 **30개 이상**이면 정규성 가정 완화됨 (중심극한정리)
- 등분산 가정이 확실하지 않으면 **Welch's t-test** 사용 권장

```mermaid
flowchart TB
    TTEST[t-검정<br/>두 그룹 평균 비교]

    TTEST --> Q1{같은 대상을<br/>두 번 측정?}

    Q1 --> |"Yes"| PAIRED[대응표본 t-검정<br/>Paired t-test<br/>ttest_rel]
    Q1 --> |"No"| IND[독립표본 t-검정<br/>Independent t-test<br/>ttest_ind]

    subgraph 독립표본["독립표본 t-검정 예시"]
        I1[A라인 vs B라인]
        I2[신규 설비 vs 기존 설비]
        I3[공급사A vs 공급사B]
    end

    subgraph 대응표본["대응표본 t-검정 예시"]
        P1[교육 전 vs 교육 후]
        P2[개선 전 vs 개선 후]
        P3[여름 vs 겨울]
    end

    IND --> 독립표본
    PAIRED --> 대응표본
```

In [7]:
# 독립표본 t-검정: Iris - Setosa vs Versicolor 꽃잎 길이

# 정규성 검정 (Shapiro-Wilk test)
print("[Step 1] 정규성 검정 (Shapiro-Wilk test)")
_, p_setosa = stats.shapiro(setosa_petal)
_, p_versi = stats.shapiro(versicolor_petal)

print(f"  Setosa p-value: {p_setosa:.4f}", end="")
print(" -> 정규분포 따름" if p_setosa >= 0.05 else " -> 정규분포 아님")
print(f"  Versicolor p-value: {p_versi:.4f}", end="")
print(" -> 정규분포 따름" if p_versi >= 0.05 else " -> 정규분포 아님")

[Step 1] 정규성 검정 (Shapiro-Wilk test)
  Setosa p-value: 0.0548 -> 정규분포 따름
  Versicolor p-value: 0.1585 -> 정규분포 따름


In [8]:
# 등분산성 검정 (Levene's test)
print("\n[Step 2] 등분산성 검정 (Levene's test)")
_, p_levene = stats.levene(setosa_petal, versicolor_petal)
print(f"  Levene p-value: {p_levene:.4f}", end="")
print(" -> 등분산" if p_levene >= 0.05 else " -> 이분산")


[Step 2] 등분산성 검정 (Levene's test)
  Levene p-value: 0.0000 -> 이분산


In [9]:
# 독립표본 t-검정 수행
print("\n[Step 3] 독립표본 t-검정 (scipy.stats.ttest_ind)")

# equal_var 파라미터: True=등분산 가정, False=이분산 가정(Welch's t-test)
t_stat, p_value = stats.ttest_ind(setosa_petal, versicolor_petal,
                                   equal_var=(p_levene >= 0.05))

print(f"\n  가설 설정:")
print(f"    H0: setosa 평균 = versicolor 평균 (평균 차이 없음)")
print(f"    H1: setosa 평균 != versicolor 평균 (평균 차이 있음)")

print(f"\n  결과:")
print(f"    t 통계량: {t_stat:.4f}")
print(f"    p-value: {p_value:.2e}")

print(f"\n  결론:", end=" ")
if p_value < 0.05:
    print("통계적으로 유의한 차이가 있음 (p < 0.05)")
    print(f"         Setosa(평균 {setosa_petal.mean():.2f}cm)와")
    print(f"         Versicolor(평균 {versicolor_petal.mean():.2f}cm)의")
    print(f"         꽃잎 길이는 유의하게 다름")
else:
    print("통계적으로 유의한 차이 없음 (p >= 0.05)")


[Step 3] 독립표본 t-검정 (scipy.stats.ttest_ind)

  가설 설정:
    H0: setosa 평균 = versicolor 평균 (평균 차이 없음)
    H1: setosa 평균 != versicolor 평균 (평균 차이 있음)

  결과:
    t 통계량: -39.4927
    p-value: 9.93e-46

  결론: 통계적으로 유의한 차이가 있음 (p < 0.05)
         Setosa(평균 1.46cm)와
         Versicolor(평균 4.26cm)의
         꽃잎 길이는 유의하게 다름


### 결과 해설

- 정규성 검정: 두 그룹 모두 정규분포를 따름
- 등분산성 검정: 두 그룹의 분산이 다름 (이분산)
- t-검정 결과: p-value가 매우 작음 (약 1e-31)
- 결론: 두 품종의 꽃잎 길이에 통계적으로 유의한 차이가 있음

**T-TEST를 위해서는, 데이터가 (1) 정규성을 만족하여야 하고 (2) 등분산성을 만족하여야 한다**

In [10]:
## 추가 예제: Titanic 요금 비교
survived_fare = titanic[titanic['survived'] == 1]['fare'].dropna()
died_fare = titanic[titanic['survived'] == 0]['fare'].dropna()

print(f"생존자 요금: 평균 {survived_fare.mean():.2f}, 중앙값 {survived_fare.median():.2f}")
print(f"사망자 요금: 평균 {died_fare.mean():.2f}, 중앙값 {died_fare.median():.2f}")

t_stat, p_value = stats.ttest_ind(survived_fare, died_fare, equal_var=False)
print(f"\nWelch's t-검정 결과:")
print(f"  t 통계량: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n  결론: 생존자와 사망자의 요금에 유의한 차이가 있음")
    print("        (비싼 티켓 구매자의 생존율이 더 높았을 수 있음)")

생존자 요금: 평균 48.40, 중앙값 26.00
사망자 요금: 평균 22.12, 중앙값 10.50

Welch's t-검정 결과:
  t 통계량: 6.8391
  p-value: 0.0000

  결론: 생존자와 사망자의 요금에 유의한 차이가 있음
        (비싼 티켓 구매자의 생존율이 더 높았을 수 있음)


## 2.4 대응표본 t-검정

### 개념 설명

동일 대상의 전후 측정값을 비교함.

| 비교 대상 | 예시 |
|----------|------|
| 교육 효과 | 같은 작업자의 교육 전후 생산성 |
| 공정 개선 | 같은 설비의 개선 전후 불량률 |
| 계절 효과 | 같은 라인의 여름/겨울 품질 |

In [11]:
# Iris 전체 데이터에서 sepal_length와 petal_length 비교
# (개념 이해용 예제)
sepal = iris['sepal_length'].values
petal = iris['petal_length'].values

print(f"꽃받침(sepal) 길이: 평균 {sepal.mean():.2f} cm")
print(f"꽃잎(petal) 길이: 평균 {petal.mean():.2f} cm")
print(f"차이: 평균 {(sepal - petal).mean():.2f} cm")

# 대응표본 t-검정
t_stat, p_value = stats.ttest_rel(sepal, petal)

print(f"\n대응표본 t-검정 (scipy.stats.ttest_rel) 결과:")
print(f"  t 통계량: {t_stat:.4f}")
print(f"  p-value: {p_value:.2e}")

if p_value < 0.05:
    print("\n  결론: 꽃받침과 꽃잎의 길이에 유의한 차이가 있음")

꽃받침(sepal) 길이: 평균 5.84 cm
꽃잎(petal) 길이: 평균 3.76 cm
차이: 평균 2.09 cm

대응표본 t-검정 (scipy.stats.ttest_rel) 결과:
  t 통계량: 22.8132
  p-value: 1.80e-50

  결론: 꽃받침과 꽃잎의 길이에 유의한 차이가 있음


## 2.5 일원분산분석 (One-way ANOVA)

**3개 이상 그룹**의 평균을 비교함.

```mermaid
flowchart TB
    ANOVA[일원분산분석<br/>One-way ANOVA<br/>3개 이상 그룹 비교]

    ANOVA --> CONCEPT[분산 분해]

    CONCEPT --> BETWEEN[그룹 간 분산<br/>Between-group<br/>집단 평균의 차이]
    CONCEPT --> WITHIN[그룹 내 분산<br/>Within-group<br/>개별 값의 산포]

    BETWEEN --> FSTAT[F-통계량<br/>= 그룹 간 분산<br/>/ 그룹 내 분산]
    WITHIN --> FSTAT

    FSTAT --> INTERPRET{F값이 크면?}

    INTERPRET --> |"큼"| DIFF[그룹 간 차이 존재<br/>p-value 작음]
    INTERPRET --> |"작음"| SAME[그룹 간 차이 없음<br/>p-value 큼]

    subgraph 가설["가설"]
        H0["H0: 모든 그룹 평균 동일"]
        H1["H1: 적어도 하나의<br/>그룹 평균이 다름"]
    end
```

In [13]:
# Wine Quality 데이터에서 품질 등급별 그룹 생성
quality_groups = []
quality_labels = sorted(wine['quality'].unique())

print(f"품질 등급별 알코올 함량:")
for q in quality_labels:
    group = wine[wine['quality'] == q]['alcohol']
    quality_groups.append(group)
    print(f"  등급 {q}: 평균 {group.mean():.2f}%, 샘플 수 {len(group)}")

품질 등급별 알코올 함량:
  등급 3: 평균 9.96%, 샘플 수 10
  등급 4: 평균 10.27%, 샘플 수 53
  등급 5: 평균 9.90%, 샘플 수 681
  등급 6: 평균 10.63%, 샘플 수 638
  등급 7: 평균 11.47%, 샘플 수 199
  등급 8: 평균 12.09%, 샘플 수 18


In [14]:
# One-way ANOVA
f_stat, p_value = stats.f_oneway(*quality_groups)

print(f"\n가설 설정:")
print(f"  H0: 모든 품질 등급의 알코올 함량 평균이 동일")
print(f"  H1: 적어도 하나의 품질 등급의 알코올 함량이 다름")

print(f"\nOne-way ANOVA (scipy.stats.f_oneway) 결과:")
print(f"  F 통계량: {f_stat:.4f}")
print(f"  p-value: {p_value:.2e}")

if p_value < 0.05:
    print("\n  결론: 품질 등급에 따라 알코올 함량에 유의한 차이가 있음")
    print("        사후검정(Post-hoc test)으로 어떤 그룹 간 차이인지 확인 필요")


가설 설정:
  H0: 모든 품질 등급의 알코올 함량 평균이 동일
  H1: 적어도 하나의 품질 등급의 알코올 함량이 다름

One-way ANOVA (scipy.stats.f_oneway) 결과:
  F 통계량: 115.8548
  p-value: 1.21e-104

  결론: 품질 등급에 따라 알코올 함량에 유의한 차이가 있음
        사후검정(Post-hoc test)으로 어떤 그룹 간 차이인지 확인 필요


- F 통계량이 크고 p-value가 매우 작음
- 결론: 와인 품질 등급에 따라 알코올 함량이 유의하게 다름
- 주의: ANOVA는 "차이가 있다"만 알려줌. 어느 그룹 간 차이인지는 사후분석 필요

## Tukey HSD 사후검정

ANOVA에서 유의한 결과가 나온 후, 어떤 그룹 간에 차이가 있는지 확인함.

- 사후분석을 통해 어느 품질 등급 간에 유의한 차이가 있는지 확인할 수 있음
- **reject=True인 쌍은 통계적으로 유의한 차이가 있는 그룹**임
- 다중비교 보정이 적용되어 1종 오류가 제어됨

In [17]:
from scipy.stats import tukey_hsd
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Tukey HSD 수행
result = tukey_hsd(*quality_groups)

print("Tukey HSD 결과 (일부 주요 비교):")
print("-" * 40)

# 결과 출력 (품질 등급 간 비교)
for i in range(len(quality_labels)):
    for j in range(i+1, len(quality_labels)):
        p_val = result.pvalue[i, j]
        if p_val < 0.05:
            print(f"등급 {quality_labels[i]} vs {quality_labels[j]}: p = {p_val:.4f} *")

print("\n* 표시: 통계적으로 유의한 차이 (p < 0.05)")

# statsmodels를 사용한 더 상세한 결과
print("\n[statsmodels를 사용한 Tukey HSD - 상세 결과]")
tukey = pairwise_tukeyhsd(wine['alcohol'], wine['quality'], alpha=0.05)
print(tukey.summary())

Tukey HSD 결과 (일부 주요 비교):
----------------------------------------
등급 3 vs 7: p = 0.0000 *
등급 3 vs 8: p = 0.0000 *
등급 4 vs 7: p = 0.0000 *
등급 4 vs 8: p = 0.0000 *
등급 5 vs 6: p = 0.0000 *
등급 5 vs 7: p = 0.0000 *
등급 5 vs 8: p = 0.0000 *
등급 6 vs 7: p = 0.0000 *
등급 6 vs 8: p = 0.0000 *

* 표시: 통계적으로 유의한 차이 (p < 0.05)

[statsmodels를 사용한 Tukey HSD - 상세 결과]
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
     3      4   0.3101 0.9231  -0.589 1.2092  False
     3      5  -0.0553    1.0  -0.886 0.7754  False
     3      6   0.6745 0.1883 -0.1566 1.5056  False
     3      7   1.5109    0.0  0.6658 2.3561   True
     3      8   2.1394    0.0  1.1109  3.168   True
     4      5  -0.3654 0.0574 -0.7373 0.0065  False
     4      6   0.3644 0.0597 -0.0084 0.7372  False
     4      7   1.2008    0.0  0.7977 1.6039   True
     4      8   1.8294    0.0  1.1179 2.5408   True
     5      6   0.7298   

## 모수 검정 요약

| 검정 | 함수 | 사용 상황 |
|------|------|----------|
| 독립표본 t-검정 | `ttest_ind()` | 두 독립 그룹 평균 비교 |
| 대응표본 t-검정 | `ttest_rel()` | 동일 대상 전후 비교 |
| 일원분산분석 | `f_oneway()` | 3개 이상 그룹 비교 |


# 범주형 데이터 검정

## 3.1 카이제곱 검정

범주형 변수 간의 관계를 분석함.

```mermaid
flowchart TB
    CHI[카이제곱 검정<br/>Chi-square Test<br/>범주형 변수 관계 분석]

    CHI --> TYPE1[독립성 검정<br/>두 범주형 변수의<br/>관계 유무]
    CHI --> TYPE2[적합도 검정<br/>관측이 이론과<br/>일치하는지]

    subgraph 독립성["독립성 검정"]
        IND1["H0: 두 변수는 독립<br/>관련 없음"]
        IND2["H1: 두 변수는 독립 아님<br/>관련 있음"]
    end

    TYPE1 --> 독립성

    subgraph 계산["계산 원리"]
        OBS[관측 빈도<br/>Observed]
        EXP[기대 빈도<br/>Expected]
        DIFF["chi-square = Sum (O-E)^2/E"]
    end
```

| 유형 | 목적 | 예시 |
|------|------|------|
| 독립성 검정 | 두 범주형 변수의 관계 | 성별 vs 생존 |
| 적합도 검정 | 관측이 이론과 일치하는지 | 불량 유형 분포 |

## 첫 단계 - 교차표 만들기

두 범주형 변수의 빈도를 집계한 표를 만들어야함


In [19]:
# Titanic - 성별과 생존여부 분석

# 교차표(Contingency Table) 생성
print("[Step 1] 교차표(Contingency Table) 생성")
contingency_table = pd.crosstab(titanic['sex'], titanic['survived'],
                                 margins=True, margins_name='Total')
contingency_table.columns = ['사망(0)', '생존(1)', 'Total']
print(contingency_table)

[Step 1] 교차표(Contingency Table) 생성
        사망(0)  생존(1)  Total
sex                        
female     81    233    314
male      468    109    577
Total     549    342    891


In [20]:
# 비율 확인
print("\n[Step 2] 성별별 생존율")
survival_rate = titanic.groupby('sex')['survived'].mean() * 100
print(f"  여성 생존율: {survival_rate['female']:.1f}%")
print(f"  남성 생존율: {survival_rate['male']:.1f}%")


[Step 2] 성별별 생존율
  여성 생존율: 74.2%
  남성 생존율: 18.9%


### 결과 해설

- 여성 생존율: 약 74.2%
- 남성 생존율: 약 18.9%
- 눈에 보이는 큰 차이가 있음. 이것이 통계적으로 유의한지 검정 필요 -> **카이제곱검정**

In [21]:
# 카이제곱 검정 수행
print("[Step 3] 카이제곱 검정 (scipy.stats.chi2_contingency)")

# margins 제외한 교차표
ct = pd.crosstab(titanic['sex'], titanic['survived'])
chi2, p_value, dof, expected = stats.chi2_contingency(ct)

print(f"\n  가설 설정:")
print(f"    H0: 성별과 생존여부는 독립 (관련 없음)")
print(f"    H1: 성별과 생존여부는 독립이 아님 (관련 있음)")

print(f"\n  결과:")
print(f"    카이제곱 통계량: {chi2:.4f}")
print(f"    자유도(df): {dof}")
print(f"    p-value: {p_value:.2e}")

print(f"\n  기대 빈도표:")
expected_df = pd.DataFrame(expected,
                           index=['female', 'male'],
                           columns=['사망(0)', '생존(1)'])
print(expected_df.round(2))

if p_value < 0.05:
    print(f"\n  결론: 성별과 생존여부는 통계적으로 유의한 관련이 있음")
    print("        여성의 생존율이 남성보다 유의하게 높았음")

[Step 3] 카이제곱 검정 (scipy.stats.chi2_contingency)

  가설 설정:
    H0: 성별과 생존여부는 독립 (관련 없음)
    H1: 성별과 생존여부는 독립이 아님 (관련 있음)

  결과:
    카이제곱 통계량: 260.7170
    자유도(df): 1
    p-value: 1.20e-58

  기대 빈도표:
         사망(0)   생존(1)
female  193.47  120.53
male    355.53  221.47

  결론: 성별과 생존여부는 통계적으로 유의한 관련이 있음
        여성의 생존율이 남성보다 유의하게 높았음


- 카이제곱 통계량: 260.72 (매우 큰 값)
- p-value: 거의 0 (약 1.2e-58)
- 결론: 성별과 생존 여부는 통계적으로 매우 강한 관련이 있음
- 기대 빈도와 관측 빈도의 차이가 매우 큼


In [22]:
# Titanic - 객실 등급(pclass)과 생존여부의 관계
ct_class = pd.crosstab(titanic['pclass'], titanic['survived'])
print("교차표:")
print(ct_class)

chi2, p_value, dof, expected = stats.chi2_contingency(ct_class)
print(f"\n카이제곱 검정 결과:")
print(f"  카이제곱 통계량: {chi2:.4f}")
print(f"  p-value: {p_value:.2e}")

print(f"\n등급별 생존율:")
for pclass in [1, 2, 3]:
    rate = titanic[titanic['pclass'] == pclass]['survived'].mean() * 100
    print(f"  {pclass}등급: {rate:.1f}%")

if p_value < 0.05:
    print(f"\n결론: 객실 등급과 생존여부는 유의한 관련이 있음")
    print("      상위 등급 승객의 생존율이 더 높았음")

교차표:
survived    0    1
pclass            
1          80  136
2          97   87
3         372  119

카이제곱 검정 결과:
  카이제곱 통계량: 102.8890
  p-value: 4.55e-23

등급별 생존율:
  1등급: 63.0%
  2등급: 47.3%
  3등급: 24.2%

결론: 객실 등급과 생존여부는 유의한 관련이 있음
      상위 등급 승객의 생존율이 더 높았음


# 비모수 검정

정규분포를 가정하지 않는 검정 방법임.

**언제 사용하는가?**
- 데이터가 **정규분포를 따르지 않을 때**
- **표본 크기가 작을 때** (n < 30)
- **순서형 데이터**일 때
- **이상치가 많을 때**

```mermaid
flowchart TB
    subgraph 모수검정["모수 검정 Parametric"]
        P1[정규분포 가정]
        P2[모수 추정<br/>평균, 분산]
        P3[검정력 높음]
        P4[대표본 유리]
    end

    subgraph 비모수검정["비모수 검정 Non-parametric"]
        NP1[분포 가정 없음]
        NP2[순위 기반]
        NP3[이상치에 강건]
        NP4[소표본 가능]
    end

    subgraph 대응관계["대응 관계"]
        M1["t-검정<br/>(독립)"] --> N1["Mann-Whitney U"]
        M2["t-검정<br/>(대응)"] --> N2["Wilcoxon"]
        M3["ANOVA"] --> N3["Kruskal-Wallis"]
    end
```

### 장단점 비교

| 장점 | 단점 |
|------|------|
| 분포 가정 불필요 | 모수 검정보다 검정력 낮음 |
| 이상치에 강건함 | 순위만 사용하여 정보 손실 |
| 표본 크기 작아도 됨 | 해석이 덜 직관적 |

## 정규성 검정 (Shapiro-Wilk)

데이터가 정규분포를 따르는지 검정함. 모수 검정과 비모수 검정 중 어떤 것을 사용할지 결정하는 데 활용됨.

**가설:**
- H0: 데이터가 정규분포를 따름
- H1: 데이터가 정규분포를 따르지 않음
- p < 0.05: 정규분포가 아님 -> 비모수 검정 사용 권장

In [24]:
from scipy.stats import shapiro

# Titanic fare 데이터의 정규성 검정
fare_data = titanic['fare'].dropna()

# 샘플 크기가 5000 초과시 오류 발생하므로 일부 샘플링
sample_fare = fare_data.sample(min(len(fare_data), 500), random_state=42)
stat, p_value = stats.shapiro(sample_fare)

print(f"Titanic 요금(fare) 정규성 검정:")
print(f"  표본 크기: {len(sample_fare)}")
print(f"  Shapiro-Wilk 통계량: {stat:.4f}")
print(f"  p-value: {p_value:.4f}")

if p_value < 0.05:
    print(f"\n  결론: 요금 데이터는 정규분포를 따르지 않음 (p < 0.05)")
    print("        비모수 검정 사용 권장")
else:
    print(f"\n  결론: 요금 데이터는 정규분포를 따름 (p >= 0.05)")

Titanic 요금(fare) 정규성 검정:
  표본 크기: 500
  Shapiro-Wilk 통계량: 0.6067
  p-value: 0.0000

  결론: 요금 데이터는 정규분포를 따르지 않음 (p < 0.05)
        비모수 검정 사용 권장


In [25]:
# Wine Quality 데이터 정규성 검정
print(f"\n\nWine Quality 주요 변수 정규성 검정:")
print("-" * 40)

wine_vars = ['alcohol', 'pH', 'residual sugar', 'quality']
for var in wine_vars:
    sample = wine[var].sample(min(len(wine), 500), random_state=42)
    stat, p_val = stats.shapiro(sample)
    normal = "정규분포" if p_val >= 0.05 else "비정규분포"
    print(f"  {var}: p = {p_val:.4f} -> {normal}")



Wine Quality 주요 변수 정규성 검정:
----------------------------------------
  alcohol: p = 0.0000 -> 비정규분포
  pH: p = 0.0105 -> 비정규분포
  residual sugar: p = 0.0000 -> 비정규분포
  quality: p = 0.0000 -> 비정규분포


## Mann-Whitney U 검정

**두 독립 그룹의 분포**를 비교하는 비모수 검정임. **독립표본 t-검정**의 비모수 대안임.

In [26]:
from scipy.stats import mannwhitneyu

# Titanic - 생존자 vs 사망자의 요금 비교 (비모수)
survived_fare = titanic[titanic['survived'] == 1]['fare'].dropna()
died_fare = titanic[titanic['survived'] == 0]['fare'].dropna()

# Mann-Whitney U 검정
stat, p_value = stats.mannwhitneyu(survived_fare, died_fare, alternative='two-sided')

print(f"생존자 요금: 중앙값 {survived_fare.median():.2f}")
print(f"사망자 요금: 중앙값 {died_fare.median():.2f}")

print(f"\nMann-Whitney U 검정 (scipy.stats.mannwhitneyu) 결과:")
print(f"  U 통계량: {stat:.2f}")
print(f"  p-value: {p_value:.4e}")

if p_value < 0.05:
    print(f"\n  결론: 생존자와 사망자의 요금 분포에 유의한 차이가 있음")

# t-검정 결과와 비교
t_stat, t_p = stats.ttest_ind(survived_fare, died_fare, equal_var=False)
print(f"\n[비교] t-검정 p-value: {t_p:.4e}")
print("       두 검정 결과가 유사하면 결론의 신뢰성이 높음")

생존자 요금: 중앙값 26.00
사망자 요금: 중앙값 10.50

Mann-Whitney U 검정 (scipy.stats.mannwhitneyu) 결과:
  U 통계량: 129951.50
  p-value: 4.5535e-22

  결론: 생존자와 사망자의 요금 분포에 유의한 차이가 있음

[비교] t-검정 p-value: 2.6993e-11
       두 검정 결과가 유사하면 결론의 신뢰성이 높음


## Wilcoxon 부호순위 검정

**대응 표본**의 비모수 검정. 대응표본 t-검정의 비모수 대안

In [27]:
from scipy.stats import wilcoxon

# Wine 데이터에서 두 산도 변수 비교
fixed = wine['fixed acidity'].values
citric = wine['citric acid'].values

# 스케일 맞추기 (min-max 정규화)
fixed_norm = (fixed - fixed.min()) / (fixed.max() - fixed.min())
citric_norm = (citric - citric.min()) / (citric.max() - citric.min())

# Wilcoxon 부호순위 검정
stat, p_value = stats.wilcoxon(fixed_norm, citric_norm)

print(f"Fixed acidity (정규화): 평균 {fixed_norm.mean():.4f}")
print(f"Citric acid (정규화): 평균 {citric_norm.mean():.4f}")

print(f"\nWilcoxon 부호순위 검정 (scipy.stats.wilcoxon) 결과:")
print(f"  통계량: {stat:.2f}")
print(f"  p-value: {p_value:.4e}")

if p_value < 0.05:
    print(f"\n  결론: 두 산도 지표 간에 유의한 차이가 있음")

Fixed acidity (정규화): 평균 0.3292
Citric acid (정규화): 평균 0.2710

Wilcoxon 부호순위 검정 (scipy.stats.wilcoxon) 결과:
  통계량: 345760.50
  p-value: 5.2319e-57

  결론: 두 산도 지표 간에 유의한 차이가 있음


## Kruskal-Wallis 검정

3개 이상 그룹의 비모수 비교임. ANOVA의 비모수 대안임.

In [28]:
from scipy.stats import kruskal

# 품질 등급별 알코올 데이터
quality_groups = [wine[wine['quality'] == q]['alcohol']
                  for q in sorted(wine['quality'].unique())]

# Kruskal-Wallis 검정
stat, p_value = stats.kruskal(*quality_groups)

print(f"Kruskal-Wallis 검정 (scipy.stats.kruskal) 결과:")
print(f"  H 통계량: {stat:.4f}")
print(f"  p-value: {p_value:.2e}")

if p_value < 0.05:
    print(f"\n  결론: 품질 등급에 따라 알코올 함량 분포에 유의한 차이가 있음")

# ANOVA와 비교
f_stat, anova_p = stats.f_oneway(*quality_groups)

print(f"\n=== 모수 vs 비모수 검정 비교 ===")
print(f"ANOVA:")
print(f"  F-통계량: {f_stat:.4f}")
print(f"  p-value: {anova_p:.2e}")

print(f"\nKruskal-Wallis:")
print(f"  H-통계량: {stat:.4f}")
print(f"  p-value: {p_value:.2e}")

print("\n두 방법 모두 유의한 결과 -> 결론 일관됨")

Kruskal-Wallis 검정 (scipy.stats.kruskal) 결과:
  H 통계량: 412.3768
  p-value: 6.37e-87

  결론: 품질 등급에 따라 알코올 함량 분포에 유의한 차이가 있음

=== 모수 vs 비모수 검정 비교 ===
ANOVA:
  F-통계량: 115.8548
  p-value: 1.21e-104

Kruskal-Wallis:
  H-통계량: 412.3768
  p-value: 6.37e-87

두 방법 모두 유의한 결과 -> 결론 일관됨


비모수 검정 요약

### 언제 사용하는가?

- 정규분포 가정을 만족하지 않을 때
- 표본 크기가 작을 때 (n < 30)
- 순서형(서열) 데이터일 때

## 검정 선택 요약

| 상황 | 정규분포 가정 | 비정규분포 |
|------|-------------|-----------|
| 2개 독립 그룹 | `ttest_ind()` | `mannwhitneyu()` |
| 2개 대응 그룹 | `ttest_rel()` | `wilcoxon()` |
| 3개+ 그룹 | `f_oneway()` | `kruskal()` |
| 범주형 vs 범주형 | `chi2_contingency()` | - |

### 판단 순서

1. 변수 유형 확인 (연속형 vs 범주형)
2. 그룹 수 확인 (2개 vs 3개 이상)
3. 정규성 검정 (`shapiro()`)
4. 적절한 검정 선택

# 핵심 정리

## 통계 검정 선택 가이드

| 상황 | 검정 방법 |
|------|----------|
| 두 그룹 평균 비교 (정규분포) | 독립표본 t-검정 (`ttest_ind`) |
| 두 그룹 평균 비교 (비정규분포) | Mann-Whitney U (`mannwhitneyu`) |
| 전후 비교 (정규분포) | 대응표본 t-검정 (`ttest_rel`) |
| 전후 비교 (비정규분포) | Wilcoxon (`wilcoxon`) |
| 3개 이상 그룹 비교 (정규분포) | ANOVA (`f_oneway`) |
| 3개 이상 그룹 비교 (비정규분포) | Kruskal-Wallis (`kruskal`) |
| 범주형 변수 관계 | 카이제곱 (`chi2_contingency`) |
| 정규성 검정 | Shapiro-Wilk (`shapiro`) |

---

## 핵심 기억사항

- **p-value < 0.05**: 귀무가설 기각 -> 유의한 차이/관계 있음
- **p-value >= 0.05**: 귀무가설 채택 -> 유의한 차이/관계 없음
- 정규성 검정 먼저! -> 모수/비모수 검정 결정
- ANOVA 유의시 -> 사후검정으로 어떤 그룹 간 차이인지 확인
- 통계적 유의성 != 실질적 중요성 (효과 크기도 고려)

---

## 자주 하는 실수

| 실수 | 올바른 접근 |
|------|------------|
| p=0.06을 "거의 유의" | 유의하지 않음 (기준 준수) |
| p값만 보고 판단 | 효과 크기도 함께 확인 |
| 여러 번 검정 후 유의한 것만 보고 | 다중비교 보정 필요 |
| 표본 작은데 모수 검정 | 비모수 검정 고려 |
| ANOVA 유의하면 끝 | 사후분석으로 어느 그룹인지 확인 |

---

## 제조업 적용 예시

| 상황 | 적용 검정 |
|------|----------|
| A/B라인 불량률 비교 | 독립표본 t-검정 |
| 공정 개선 전후 비교 | 대응표본 t-검정 |
| 3개 교대조 생산성 비교 | ANOVA |
| 라인별 불량 유형 분포 | 카이제곱 검정 |
| 소량 샘플 품질 비교 | Mann-Whitney U |